# Modeling starter — CNN Auto-Tagger (Model 1)

Kerangka untuk temanmu agar bisa langsung mulai modeling memakai output
`preprocessing.ipynb`. Isi sel-sel yang bertanda **TODO** sesuai kebutuhan.

Kontrak WAJIB (PRD §4.4.3):
- Input `image` bentuk `(B,128,128,3)`, nilai **0–255** (normalisasi di dalam model).
- Layer `embedding` = `Dense(256, relu)`.
- Head output bernama `article_type`, `base_colour`, `usage` (softmax).
- Urutan kelas = urutan di `ml/configs/labels.yaml` (jangan diubah tanpa retrain).

## 0. Setup

In [1]:
import os, sys, json
from pathlib import Path
from collections import Counter

# Pastikan working directory = folder ml/ (tempat config_utils.py berada).
HERE = Path.cwd()
if not (HERE / "config_utils.py").exists():
    if (HERE / "ml" / "config_utils.py").exists():
        os.chdir(HERE / "ml")
    else:
        raise RuntimeError("Buka/run notebook ini dari folder ml/.")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (12, 4)

import config_utils as cu

IMG_SIZE = 128
BATCH_SIZE = 128
SEED = 42
AUTOTUNE = None  # di-set setelah import tensorflow
RUN_TRAINING = False   # ubah ke True untuk mulai melatih

## 1. Load label & hasil preprocessing

In [2]:
import tensorflow as tf
AUTOTUNE = tf.data.AUTOTUNE

labels = cu.load_labels()
article_keys = [e["key"] for e in labels["article_type"]]
colour_keys = [e["key"] for e in labels["base_colour"]]
usage_keys = [e["key"] for e in labels["usage"]]
a2i = {k: i for i, k in enumerate(article_keys)}
c2i = {k: i for i, k in enumerate(colour_keys)}
u2i = {k: i for i, k in enumerate(usage_keys)}

PROCESSED = cu.fashion_processed_dir()
images = np.load(PROCESSED / "fashion_128.npy", mmap_mode="r")
print("images:", images.shape, images.dtype)
print("labels  : article=%d colour=%d usage=%d" % (len(article_keys), len(colour_keys), len(usage_keys)))

images: (33621, 128, 128, 3) uint8
labels  : article=24 colour=14 usage=5


## 2. `tf.data` dari split CSV

In [3]:
def _load_image(index):
    return images[index].astype(np.float32)


def make_dataset(split, batch_size=BATCH_SIZE, training=False):
    df = pd.read_csv(PROCESSED / "splits" / f"{split}.csv")
    idx = df["npy_index"].to_numpy()
    y_art = df["article_type"].map(a2i).to_numpy(dtype=np.int32)
    y_col = df["base_colour"].map(c2i).to_numpy(dtype=np.int32)
    y_usg = df["usage"].replace("", np.nan).map(u2i).fillna(0).astype(np.int32).to_numpy()
    usg_w = df["usage_valid"].astype(np.float32).to_numpy()

    ds = tf.data.Dataset.from_tensor_slices((idx, y_art, y_col, y_usg, usg_w))
    if training:
        ds = ds.shuffle(min(len(df), 10000), seed=SEED, reshuffle_each_iteration=True)

    def _map(i, ya, yc, yu, uw):
        img = tf.numpy_function(_load_image, [i], tf.float32)
        img.set_shape((IMG_SIZE, IMG_SIZE, 3))
        y = {"article_type": ya, "base_colour": yc, "usage": yu}
        w = {"article_type": tf.ones_like(uw), "base_colour": tf.ones_like(uw), "usage": uw}
        return img, y, w

    return ds.map(_map, num_parallel_calls=AUTOTUNE).batch(batch_size).prefetch(AUTOTUNE)


train_ds = make_dataset("train", training=True)
x_batch, y_batch, w_batch = next(iter(train_ds))
print("x:", x_batch.shape, x_batch.dtype, "range", float(x_batch.numpy().min()), float(x_batch.numpy().max()))
print("y:", {k: tuple(v.shape) for k, v in y_batch.items()})
print("usage weight mean:", float(w_batch["usage"].numpy().mean()))

x: (128, 128, 128, 3) <dtype: 'float32'> range 0.0 255.0
y: {'article_type': (128,), 'base_colour': (128,), 'usage': (128,)}
usage weight mean: 0.921875


## 3. Arsitektur (MobileNetV2 multi-output)

In [4]:
import keras
from keras import layers

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.04),
    layers.RandomZoom((-0.15, 0.15)),
    layers.RandomTranslation(0.08, 0.08, fill_mode="reflect"),
    layers.RandomBrightness(0.25, value_range=(0, 255)),
    layers.RandomContrast(0.25),
], name="augment")


def build_tagger(num_article, num_colour, num_usage, emb_dim=256, dropout=0.3, pretrained="imagenet"):
    inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")
    x = data_augmentation(inp)
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="preprocess")(x)
    backbone = keras.applications.MobileNetV2(
        include_top=False, weights=pretrained, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    backbone.trainable = False
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(dropout)(x)
    emb = layers.Dense(emb_dim, activation="relu", name="embedding")(x)
    z = layers.Dropout(dropout)(emb)
    out = {
        "article_type": layers.Dense(num_article, activation="softmax", name="article_type")(z),
        "base_colour": layers.Dense(num_colour, activation="softmax", name="base_colour")(z),
        "usage": layers.Dense(num_usage, activation="softmax", name="usage")(z),
    }
    return keras.Model(inp, out, name="sw_tagger"), backbone, emb


model, backbone, embedding_tensor = build_tagger(len(article_keys), len(colour_keys), len(usage_keys))
model.summary()

# Model inferensi 4-output (dipakai 05_export.py / backend): 3 head + embedding.
infer_model = keras.Model(
    model.input,
    {**{k: model.output[k] for k in ("article_type", "base_colour", "usage")},
     "embedding": embedding_tensor},
    name="tagger_infer",
)
print("infer_model outputs:", list(infer_model.output.keys()) if isinstance(infer_model.output, dict) else infer_model.output)

Model: "sw_tagger"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 128, 128,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ augment             │ (None, 128, 128,  │          0 │ image[0][0]       │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ preprocess          │ (None, 128, 128,  │          0 │ augment[0][0]     │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_1… │ (None, 4, 4,      │  2,257,984 │ preprocess[0][0]  │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gap                 │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 1280)      │          0 │ gap[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding (Dense)   │ (None, 256)       │    327,936 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ article_type        │ (None, 24)        │      6,168 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ base_colour (Dense) │ (None, 14)        │      3,598 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ usage (Dense)       │ (None, 5)         │      1,285 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,596,971 (9.91 MB)

 Trainable params: 338,987 (1.29 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

infer_model outputs: ['article_type', 'base_colour', 'usage', 'embedding']


## 4. Compile & latih

In [5]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss={h: "sparse_categorical_crossentropy" for h in ("article_type", "base_colour", "usage")},
    loss_weights={"article_type": 1.0, "base_colour": 0.6, "usage": 0.4},
    metrics={h: ["accuracy"] for h in ("article_type", "base_colour", "usage")},
)

if RUN_TRAINING:
    val_ds = make_dataset("val")
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2),
        keras.callbacks.ModelCheckpoint("artifacts/tagger_best.keras", save_best_only=True),
    ]
    history = model.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=callbacks)
    infer_model.save("artifacts/tagger_infer.keras")
    print("model disimpan ke artifacts/tagger_infer.keras")
else:
    print("RUN_TRAINING=False. Set True untuk mulai training (disarankan GPU/Colab).")

RUN_TRAINING=False. Set True untuk mulai training (disarankan GPU/Colab).


## 5. TODO untuk temanmu

- Phase 2: unfreeze 30 layer terakhir backbone (kecuali `BatchNormalization`), `Adam(2e-5)`.
- Tambah *class weight* akar-terbalik & *oversampling* kelas minoritas (`skirt`, `jacket`).
- Evaluasi test split: akurasi + `classification_report` (macro-F1) → `artifacts/metrics.json`.
- Validasi bahwa `preprocess_pil` tetap identik dengan backend sebelum ekspor.